# Goal
I want to see a practical example of how gradient checking works. From this practical example I want to gain an insight of how I can use gradient checking to confirm my work. 

*(On a side tangent, I will see how non-normalized inputs affect gradient descent, particularly if calculated derivatives are off)*

1. The first step is to implement gradient descent and confirm grad checking with a simple one unit network
2. I'll then expand to a 2 layer network
3. Finally, I'll modify the 2 layer network to look similar to my network

In [156]:
# Imports
import numpy as np
import numpy.typing as npt

In [157]:
# Dummy dataset
X = np.array([
    [
        [100],
        [2]    
    ],
    [
        [50],
        [40]    
    ],
    [
        [70],
        [30]    
    ],
    [
        [400],
        [2]
    ]
])
Y = np.array([1205, 4505, 3705, 4205])
print(X.shape)
print(Y.shape)

(4, 2, 1)
(4,)


In [158]:
# J_theta that will be used by all networks
def J_theta(
    X: npt.NDArray,
    Y: npt.NDArray,
    f_x,
    theta: npt.NDArray
):
    """ 
    Return the cost J for condensed parameters w for given dataset
    
    Args:
        X (ndarray): a (m, 2, 1) array with m training features
        Y (ndarray): a (m,) array with m target labels
        f_x: function to get prediction for training example
        theta (ndarray): array with condensed parameters
        
    Returns:
        J (scalar): cost
    """ 
    m = X.shape[0]
    J = 0
    
    for i in range(m):
        x_i = X[i]
        y_i = Y[i]
        y_pred_i = f_x(
            x_i,
            theta
        )
        J += (y_pred_i - y_i) ** 2
        
    return J / (2 * m)

In [159]:
# Grad check that will be used by all networks
def grad_check(
    X: npt.NDArray,
    Y: npt.NDArray,
    theta: npt.NDArray,
    f_x,
    calculate_gradients
):
    """ 
    Gets approximate dtheta and compares it with dtheta gotten with implementation of gradient descent
    
    Args:
        X (ndarray): a (m, 2, 1) array with m training examples
        Y (ndarray): a (m,) array with labels for each training example
        theta (ndarray): condensed parameters for network
        f_x: function to make prediction given inputs and condensed parameters
        calculate_gradients: function to return dtheta for parameters
        
    Returns:
        similarity_check (scalar): value with distance between dtheta approx and dtheta
        dtheta_approx (ndarray): condensed approximate derivative
        dtheta (ndarray): condensed derivatives
    """
    num_params = theta.shape[0]
    dtheta = calculate_gradients(
        X,
        Y,
        theta
    )
    dtheta_approx = np.zeros((num_params,))
    epsilon = 1e-7
    
    for i in range(num_params):
        theta_right = theta.copy()
        theta_left = theta.copy()
        theta_right[i] += epsilon
        theta_left[i] -= epsilon
        J_theta_right = J_theta(X,Y,f_x,theta_right)
        J_theta_left = J_theta(X,Y,f_x,theta_left)
        d_theta_approx_i = (J_theta_right - J_theta_left) / (2 * epsilon)
        dtheta_approx[i] = d_theta_approx_i
       
    print(dtheta) 
    print(dtheta_approx)
    
    distance = np.linalg.norm(dtheta_approx - dtheta)
    similarity_check = distance / (np.linalg.norm(dtheta_approx) + np.linalg.norm(dtheta))
    
    # Interpreting results
    print(f"The difference between dtheta_approx and dtheta is {similarity_check}")
    return (similarity_check, dtheta_approx, dtheta)


# Simple network

In [160]:
def f_x_simple(
    x_i: npt.NDArray,
    theta: npt.NDArray
):
    """ 
    Function to make prediction based on the given inputs and parameters. Prediction is wx + b
    
    Args:
        x_i (ndarray): a (2,1) array
        theta: a (3,) array with condensed parameters
        
    Returns:
        y_pred (scalar): prediction of the model
    """
    w = np.array([
        [theta[0], theta[1]]
    ])
    b = theta[2]
    
    return (np.matmul(w, x_i) + b)[0][0]

print(f_x_simple(np.array([
    [100],
    [2]
]), np.array([10, 100, 5])))
print(J_theta(
    X=X,
    Y=Y,
    f_x=f_x_simple,
    theta=np.array([0,0,0])
))

1205
6644512.5


In [161]:
def calculate_gradients_simple(
    X: npt.NDArray,
    Y: npt.NDArray,
    theta: npt.NDArray
):
    """ 
    Function to get dw and db for our simple example that will be used to minimize the cost function
    
    Args:
        X (ndarray): a (m, 2, 1) array with training features
        Y (ndarray): a (m, ) array with target outputs
        theta (ndarray): a (3,) array with condensed parameters of network
        
    Returns
        dtheta (ndarray): a (3,) array with condensed derivatives of parameters
    """
    m = X.shape[0]
    dw = np.zeros((1, 2))
    db = 0
    
    for i in range(m):
        x_i = X[i]
        y_i = Y[i]
        y_pred_i = f_x_simple(
            x_i=x_i,
            theta=theta
        )
        
        dw += (y_pred_i - y_i) * x_i.T # Multplying by the parameter of each input
        db += y_pred_i - y_i
        
    dw /= m
    db /= m    
    dtheta = np.array([dw[0][0], dw[0][1], db])
    return dtheta

calculate_gradients_simple(
    X=X,
    Y=Y,
    theta=np.array([0,0,0])
)

array([-571775. ,  -75542.5,   -3405. ])

In [162]:
def analyze_grad_check_simple(
    X: npt.NDArray,
    Y: npt.NDArray,
    theta: npt.NDArray,
    calculate_gradients
):
    """ 
    Gets approximate dtheta and compares it with dtheta gotten with implementation of gradient descent
    
    Args:
        X (ndarray): a (m, 2, 1) array with m training examples
        Y (ndarray): a (m,) array with labels for each training example
        theta (ndarray): condensed parameters for network
        calculate_gradients: function to calculate gradients
    """
    _, dtheta_approx, dtheta = grad_check(
        X=X,
        Y=Y,
        f_x=f_x_simple,
        calculate_gradients=calculate_gradients,
        theta=theta
    )
    
    w_problem = 0
    b_problem = 0
    
    # Getting total problem for each category
    for i in range(3):
        dtheta_value = dtheta[i]
        dtheta_approx_value = dtheta_approx[i]
        difference = (dtheta_approx_value - dtheta_value) ** 2
        if 0 <= i < 2:
            w_problem += difference / (np.abs(dtheta_approx_value) ** 2  + np.abs(dtheta_value) ** 2)
        elif 2 <= i:
            b_problem += difference / (np.abs(dtheta_approx_value) ** 2  + np.abs(dtheta_value) ** 2)
        else:
            print(f"Don't know how to intepret {i}")
            
    print(f"Problem for b is {b_problem}")
    print(f"Problem for w is {w_problem}")
    
    for idx in range(3):
        dtheta_value = dtheta[idx]
        dtheta_approx_value = dtheta_approx[idx]
        if 0 <= idx < 2:
            print(f"Element index {idx} in w has dtheta of {dtheta_value} and dtheta_approx of {dtheta_approx_value}")
        elif 2 <= idx:
            print(f"Element index {idx} in b has dtheta of {dtheta_value} and dtheta_approx of {dtheta_approx_value}")
        else:
            print(f"Don't know how to intepret {idx}")


In [163]:
analyze_grad_check_simple(
    X=X,
    Y=Y,
    theta=np.random.rand(3,),
    calculate_gradients=calculate_gradients_simple
)

[-558114.35287088  -74968.84199596   -3352.92620021]
[-558114.35449868  -74968.8455835    -3352.91959345]
The difference between dtheta_approx and dtheta is 6.82974935017806e-09
Problem for b is 1.941330983830169e-12
Problem for w is 1.1492446069585792e-15
Element index 0 in w has dtheta of -558114.3528708773 and dtheta_approx of -558114.3544986844
Element index 1 in w has dtheta of -74968.84199595726 and dtheta_approx of -74968.84558349848
Element index 2 in b has dtheta of -3352.9262002107707 and dtheta_approx of -3352.9195934534073


In [164]:
def calculate_gradients_simple_wrong(
    X: npt.NDArray,
    Y: npt.NDArray,
    theta: npt.NDArray
):
    """ 
    Function to get dw and db for our simple example that will be used to minimize the cost function
    
    Args:
        X (ndarray): a (m, 2, 1) array with training features
        Y (ndarray): a (m, ) array with target outputs
        theta (ndarray): a (3,) array with condensed parameters of network
        
    Returns
        dtheta (ndarray): a (3,) array with condensed derivatives of parameters
    """
    m = X.shape[0]
    dw = np.zeros((1, 2))
    db = 0
    
    for i in range(m):
        x_i = X[i]
        y_i = Y[i]
        y_pred_i = f_x_simple(
            x_i=x_i,
            theta=theta
        )
        
        dw += (y_pred_i - y_i) * x_i.T
        db += (y_pred_i - y_i) + 1 # Making db wrong
        
    dw /= m
    db /= m    
    dtheta = np.array([dw[0][0], dw[0][1], db])
    return dtheta

calculate_gradients_simple_wrong(
    X=X,
    Y=Y,
    theta=np.array([0,0,0])
)

array([-571775. ,  -75542.5,   -3404. ])

In [165]:
analyze_grad_check_simple(
    X=X,
    Y=Y,
    theta=np.random.rand(3,),
    calculate_gradients=calculate_gradients_simple_wrong
)

[-555759.22741829  -74516.87755317   -3334.47832784]
[-555759.23062861  -74516.87473804   -3335.48057824]
The difference between dtheta_approx and dtheta is 8.936895387476762e-07
Problem for b is 4.515815105454287e-08
Problem for w is 7.302886763752631e-16
Element index 0 in w has dtheta of -555759.2274182858 and dtheta_approx of -555759.2306286097
Element index 1 in w has dtheta of -74516.8775531682 and dtheta_approx of -74516.87473803759
Element index 2 in b has dtheta of -3334.4783278367704 and dtheta_approx of -3335.4805782437325


# Lessons learnt
1. Grad check does lead to results that are very similar to the known derivative
2. Use numpy numbers for theta, I think standard python numbers don't have precision needed for these operations to work well i.e. either initialize with np.rand.random or np.zeros or something like that
3. I should calculate problem for each parameter in the same way I do it for all parameters

```
The difference between dtheta_approx and dtheta is 6.1156563530755895e-09
Problem for b is 2.926869964284918e-13
Problem for w is 6.554599753018853e-17
Element index 0 in w has dtheta of -545571.7506854219 and dtheta_approx of -545571.7444419861 with difference of 0.006243435782380402
Element index 1 in w has dtheta of -74226.14374310365 and dtheta_approx of -74226.14376991987 with difference of -2.681622572708875e-05
Element index 2 in b has dtheta of -3299.9108443650775 and dtheta_approx of -3299.9133691191673 with difference of -0.002524754089790804
```

Definetly the difference between dtheta_approx and dtheta is a good indicator of whether something in backprop is wrong. But if the difference in one of the parameters is small (especially one with few values) the effect won't be visible.

In general, if the difference for an individual parameter is less than e-12 then backprop for that parameter has been implemented correctly

The difference of individual numbers i.e (x - y) should not be used as the only indicator.



# 2 layer network

Layer 1 will have 2 units and the output layer will have one unit

In [166]:
def f_x_2_layer(
    x_i: npt.NDArray,
    theta: npt.NDArray
):
    """ 
    Return prediction from simple 2 layer network given input and condensed parameters
    
    Args:
        x_i (ndarray): a (2, 1) array with training features
        theta (ndarray): a (9,) array with condensed parameters
        
    Returns:
        y_pred (scalar): prediction of the network,
        z_2 (ndarray): cached z_2 value
        a_1 (ndarray): cached a_1 value
        z_1 (ndarray): cached z_1 value
    """
    w_1 = np.array(theta[0:4]).reshape((2,2))
    b_1 = np.array(theta[4:6]).reshape((2, 1))
    w_2 = np.array(theta[6:8]).reshape((1, 2))
    b_2 = theta[8]
    
    z_1 = np.matmul(w_1, x_i) + b_1
    a_1 = np.maximum(0, z_1)
    z_2 = np.matmul(w_2, a_1) + b_2
    a_2 = np.maximum(0, z_2)
    
    return (a_2[0][0], z_2, a_1, z_1)
    
print(f_x_2_layer(np.array([
    [100],
    [2]
]), np.array([1,2,3,4,5,6,7,8,9]))[0])

3284


In [167]:
def calculate_gradients_2_layer(
    X: npt.NDArray,
    Y: npt.NDArray,
    theta: npt.NDArray
):
    """ 
    Return condensed derivatives of the network based on given compressed inputs
    
    Args:
        X (ndarray): a (m, 2, 1) array with m training examples
        Y (ndarray): a (m,) array with target outputs
        theta (ndarray): a (9,) array with condensed parameters
        
    Returns:
        dtheta (ndarray): a (9,) array with condensed derivatives
    """
    m = X.shape[0]
    dw_1 = np.zeros((2,2))
    db_1 = np.zeros((2,1))
    dw_2 = np.zeros((1,2))
    db_2 = 0
    
    w_2 = np.array(theta[6:8]).reshape((1, 2))
    
    for i in range(m):
        x_i = X[i]
        y_i = Y[i]
        y_pred_i, z_2_i, a_1_i, z_1_i = f_x_2_layer(
            x_i=x_i,
            theta=theta
        )
        
        dy_pred_i = np.array([[(y_pred_i - y_i)]])
        dz_2_i = np.where(z_2_i < 0, np.zeros((1,1)), dy_pred_i)
        dw_2 += np.matmul(dz_2_i, a_1_i.T)
        db_2 += dz_2_i[0][0]
        da_1_i = np.matmul(w_2.T, dz_2_i)
        dz_1_i = np.where(z_1_i < 0, np.zeros((2,1)), da_1_i)
        dw_1 += np.matmul(dz_1_i, x_i.T)
        db_1 += dz_1_i
    
    dw_1 /= (m)
    dw_2 /= (m)
    db_1 /= (m) 
    db_2 /= (m)
    
    dtheta = []
    dtheta.extend(dw_1.flatten())
    dtheta.extend(db_1.flatten())
    dtheta.extend(dw_2.flatten())
    dtheta.append(db_2)
    return np.array(dtheta)

In [168]:
calculate_gradients_2_layer(
    X=X,
    Y=Y,
    theta=np.array([1,2,3,4,5,6,7,8,9])
)

array([ 6.131895e+06, -3.831450e+04,  7.007880e+06, -4.378800e+04,
        1.640100e+04,  1.874400e+04,  8.767530e+05,  2.620119e+06,
        2.343000e+03])

In [169]:
def analyze_grad_check_2_layer(
    X: npt.NDArray,
    Y: npt.NDArray,
    theta: npt.NDArray,
    calculate_gradients
):
    """ 
    Gets approximate dtheta and compares it with dtheta gotten with implementation of gradient descent
    
    Args:
        X (ndarray): a (m, 2, 1) array with m training examples
        Y (ndarray): a (m,) array with labels for each training example
        theta (ndarray): condensed parameters for network
        calculate_gradients: function to return dtheta for parameters
    """
    _, dtheta_approx, dtheta = grad_check(
        X=X,
        Y=Y,
        theta=theta,
        f_x= lambda x_i, theta: f_x_2_layer(x_i=x_i, theta=theta)[0],
        calculate_gradients=calculate_gradients
    )
    
    w_1_problem = 0
    b_1_problem = 0
    w_2_problem = 0
    b_2_problem = 0
    
    # Getting total problem for each category
    for i in range(9):
        dtheta_value = dtheta[i]
        dtheta_approx_value = dtheta_approx[i]
        difference = (dtheta_approx_value - dtheta_value) ** 2
        
        if 0 <= i < 4:
            w_1_problem += difference / (np.abs(dtheta_approx_value) ** 2  + np.abs(dtheta_value) ** 2)
        elif 4 <= i < 6:
            b_1_problem += difference / (np.abs(dtheta_approx_value) ** 2  + np.abs(dtheta_value) ** 2)
        elif 6 <= i < 8:
            w_2_problem += difference / (np.abs(dtheta_approx_value) ** 2  + np.abs(dtheta_value) ** 2)
        elif 8 <= i:
            b_2_problem += difference / (np.abs(dtheta_approx_value) ** 2  + np.abs(dtheta_value) ** 2)
        else:
            print(f"Don't know how to intepret {i}")
            
    print(f"Problem for w_1 is {w_1_problem}")
    print(f"Problem for b_1 is {b_1_problem}")
    print(f"Problem for w_2 is {w_2_problem}")
    print(f"Problem for b_2 is {b_2_problem}")
    
    for idx in range(9):
        dtheta_value = dtheta[idx]
        dtheta_approx_value = dtheta_approx[idx]
        
        if 0 <= idx < 4:
            print(f"Element index {idx} in w_1 has dtheta of {dtheta_value} and dtheta_approx of {dtheta_approx_value}")
        elif 4 <= idx < 6:
            print(f"Element index {idx} in b_1 has dtheta of {dtheta_value} and dtheta_approx of {dtheta_approx_value}")
        elif 6 <= idx < 8:
            print(f"Element index {idx} in w_2 has dtheta of {dtheta_value} and dtheta_approx of {dtheta_approx_value}")
        elif 8 <= idx:
            print(f"Element index {idx} in b has dtheta of {dtheta_value} and dtheta_approx of {dtheta_approx_value}")
        else:
            print(f"Don't know how to intepret {idx}")

In [170]:
analyze_grad_check_2_layer(
    X=X,
    Y=Y,
    theta=np.random.rand(9,),
    calculate_gradients=calculate_gradients_2_layer
)

[ -72786.67169879   -9686.83401974  -83679.87994205  -11136.55960455
    -435.12248553    -500.24264745 -195735.60228956 -359475.43302354
   -3383.38723068]
[ -72786.67297214   -9686.82579696  -83679.87815291  -11136.56442612
    -435.12322009    -500.25060773 -195735.59518903 -359475.43568909
   -3383.38315487]
The difference between dtheta_approx and dtheta is 1.8013894527878785e-08
Problem for w_1 is 4.5438741070051065e-13
Problem for b_1 is 1.2803216709496797e-10
Problem for w_2 is 6.854697527936508e-16
Problem for b_2 is 7.255966172169963e-13
Element index 0 in w_1 has dtheta of -72786.67169878754 and dtheta_approx of -72786.6729721427
Element index 1 in w_1 has dtheta of -9686.834019739013 and dtheta_approx of -9686.825796961784
Element index 2 in w_1 has dtheta of -83679.87994204662 and dtheta_approx of -83679.8781529069
Element index 3 in w_1 has dtheta of -11136.55960454909 and dtheta_approx of -11136.564426124096
Element index 4 in b_1 has dtheta of -435.12248553286713 and dt

# Lessons Learnt

1. Issues with how the gradient of a layer L is calculated have a great effect on layers M where M < L. 
2. Logging dtheta approx and dtheta values can sometimes give you a hint of what the problem could be

This means that you should prioritize fixing gradient issues of layers that are closer to the output layer.

Also gradient checking should be done before tuning any gradient descent operators